<a href="https://colab.research.google.com/github/thekingsdev/edge_detection/blob/main/CSKP_EDGE_DECTECTION_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from IPython.display import display
import io
from PIL import Image

def preprocess_image(img):
    """Preprocesses the image for edge detection."""
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img

    # CLAHE for contrast enhancement in medical imaging
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    # Gaussian Blur for noise reduction
    blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)

    return blurred.astype(np.float32) / 255.0

def compute_coefficients(t):
    """Computes coefficients a1, a2, a3 based on the Sakaguchi-type model."""
    t = max(0, min(t, 0.999))
    a1 = 1.0
    a2 = 1.0 / (1.0 - t)
    term1 = 1.0 / (2.0 - t - t**2)
    term2 = max(1.0, (1.0 + t) / (1.0 - t))
    a3 = term1 * term2
    return a1, a2, a3

def generate_masks(a1, a2, a3):
    """Generates all 8 directional masks M1-M8 explicitly."""
    # Horizontal and Vertical
    m1 = np.array([[-a3, -a2, -a3], [0, 0, 0], [a3, a2, a3]])
    m3 = np.array([[a3, 0, -a3], [a2, 0, -a2], [a3, 0, -a3]])
    m5 = np.array([[a3, a2, a3], [0, 0, 0], [-a3, -a2, -a3]])
    m7 = np.array([[-a3, 0, a3], [-a2, 0, a2], [-a3, 0, a3]])

    # Diagonals
    m2 = np.array([[0, -a3, -a2], [a3, 0, -a3], [a2, a3, 0]])
    m4 = np.array([[a2, a3, 0], [a3, 0, -a3], [0, -a3, -a2]])
    m6 = np.array([[0, a3, a2], [-a3, 0, a3], [-a2, -a3, 0]])
    m8 = np.array([[-a2, -a3, 0], [-a3, 0, a3], [0, a3, a2]])

    return [m1, m2, m3, m4, m5, m6, m7, m8]

def apply_cskp(image, t):
    """Convolves the image with all 8 masks and returns the max response."""
    a1, a2, a3 = compute_coefficients(t)
    masks = generate_masks(a1, a2, a3)

    responses = []
    for mask in masks:
        res = cv2.filter2D(image, -1, mask)
        responses.append(np.abs(res))

    return np.max(np.stack(responses), axis=0)

def start_app(uploaded_file):
    if not uploaded_file:
        print("Please upload a file first.")
        return

    content = next(iter(uploaded_file.values()))['content']
    img = np.array(Image.open(io.BytesIO(content)))
    prep = preprocess_image(img)

    def interactive_viewer(t, threshold):
        cskp_res = apply_cskp(prep, t)
        _, binary = cv2.threshold(cskp_res, threshold, 1.0, cv2.THRESH_BINARY)
        canny = cv2.Canny((prep*255).astype(np.uint8), 100, 200)

        fig, ax = plt.subplots(1, 4, figsize=(24, 7))
        ax[0].imshow(prep, cmap='gray'); ax[0].set_title("Preprocessed")
        ax[1].imshow(cskp_res, cmap='magma'); ax[1].set_title(f"CSKP Magnitude (t={t:.2f})")
        ax[2].imshow(binary, cmap='gray'); ax[2].set_title("CSKP Binary Edge")
        ax[3].imshow(canny, cmap='gray'); ax[3].set_title("Standard Canny")
        for a in ax: a.axis('off')
        plt.show()

        a1, a2, a3 = compute_coefficients(t)
        print(f"Parameters: t={t:.2f} | a1={a1:.2f}, a2={a2:.2f}, a3={a3:.4f}")

    interact(interactive_viewer,
             t=widgets.FloatSlider(min=0.0, max=0.95, step=0.01, value=0.5, description='CSKP t'),
             threshold=widgets.FloatSlider(min=0.0, max=0.8, step=0.01, value=0.15, description='Threshold'))

In [11]:
uploader = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
btn = widgets.Button(description="Generate Edges", button_style='primary')

def on_click(b):
    if uploader.value:
        start_app(uploader.value)
    else:
        print("Click Upload first!")

btn.on_click(on_click)
display(widgets.VBox([widgets.Label("Upload Mandibular X-ray:"), uploader, btn]))

interactive(children=(FloatSlider(value=0.5, description='CSKP t', max=0.95, step=0.01), FloatSlider(value=0.1…

interactive(children=(FloatSlider(value=0.5, description='CSKP t', max=0.95, step=0.01), FloatSlider(value=0.1…